In [0]:
# reading table bronze to silver 

df=spark.table("customer_pipeline.customer_data.bronze_customers")
display(df)

customer_id,customer_name,city,email,age,salary
1001,Rahul,Bangalore,rahul@gmail.com,25.0,42000.0
1002,Arun,Chennai,arun@gmail.com,28.0,51000.0
1003,Priya,Hyderabad,priya@gmail.com,24.0,46000.0
1004,Sneha,Bangalore,sneha@gmail.com,27.0,48000.0
1005,Kiran,Mumbai,kiran@gmail.com,31.0,62000.0
1006,Meena,Pune,meena@gmail.com,29.0,55000.0
1007,Vijay,Bangalore,vijay@gmail.com,32.0,58000.0
1008,Anita,Delhi,anita@gmail.com,26.0,45000.0
1009,Ravi,Chennai,ravi@gmail.com,30.0,52000.0
1010,Divya,Hyderabad,divya@gmail.com,23.0,39000.0


In [0]:
# Your Silver layer tasks:
# 1. Read data from bronze_customers.
# 2. Remove duplicate customer IDs.
# 3. Replace NULL city with Unknown.
# 4. Remove extra spaces from customer names.
# 5. Identify invalid email addresses.
# 6. Handle NULL age and salary.
# 7. Save clean records into silver_customers.
# 8. Save rejected records into rejected_customers.

In [0]:
# Task 2: Remove duplicate customer records

from pyspark.sql.functions import count

df.groupBy("customer_id")\
    .agg(count("*").alias("total_count"))\
    .filter("total_count > 1")\
    .show()


+-----------+-----------+
|customer_id|total_count|
+-----------+-----------+
|       1005|          2|
+-----------+-----------+



In [0]:
# remove duplicates based on customer_id

df=df.dropDuplicates(["customer_id"])
print("Records after removing duplicates: ",df.count())

Records after removing duplicates:  25


In [0]:
# Task 3: Replace NULL city with Unknown
from pyspark.sql.functions import col,when
df=df.withColumn("city",
              when(col("city").isNull(),"unknown").otherwise(col("city"))
              )

df.filter(col("city").isNull()).show()

+-----------+-------------+----+-----+---+------+
|customer_id|customer_name|city|email|age|salary|
+-----------+-------------+----+-----+---+------+
+-----------+-------------+----+-----+---+------+



In [0]:
# Task 4: Remove extra spaces from customer names
from pyspark.sql.functions import trim
df=df.withColumn("customer_name",trim(col("customer_name")))

In [0]:
# Task 5: Identify invalid email addresses
df = df.withColumn(
    "email_status",
    when(
        col("email").rlike(r"^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}$"),
        "Valid"
    ).otherwise("Invalid")
)
df.filter(col("email_status") == "Invalid").show()

+-----------+-------------+---------+--------------+----+-------+------------+
|customer_id|customer_name|     city|         email| age| salary|email_status|
+-----------+-------------+---------+--------------+----+-------+------------+
|       1018|       Geetha|Hyderabad|          NULL|26.0|44000.0|     Invalid|
|       1020|        Deepa|   Mumbai|deepagmail.com|25.0|41000.0|     Invalid|
|       1021|       Naveen|     Pune|       naveen@|30.0|56000.0|     Invalid|
+-----------+-------------+---------+--------------+----+-------+------------+



In [0]:
# Task 6: Handle NULL age and salary
df.filter(
    col("age").isNull() | col("salary").isNull()
).show()

+-----------+-------------+---------+----------------+----+-------+------------+
|customer_id|customer_name|     city|           email| age| salary|email_status|
+-----------+-------------+---------+----------------+----+-------+------------+
|       1019|       Sanjay|Bangalore|sanjay@gmail.com|NULL|59000.0|       Valid|
|       1022|       Swathi|Bangalore|swathi@gmail.com|24.0|   NULL|       Valid|
+-----------+-------------+---------+----------------+----+-------+------------+



In [0]:
# For this project, we'll use:
# - NULL age → 0
# - NULL salary → 0.0

df=df.withColumn(
    "age",
    when(col("age").isNull(),0).otherwise(col("age"))
)
df=df.withColumn(
    "salary",
    when(col("salary").isNull(),0.0).otherwise(col("salary"))
)

In [0]:
# Task 7: Separate invalid email records
# We'll keep valid emails in Silver and invalid emails in the rejected table.
valid_df = df.filter(col("email_status") == "Valid")
rejected_df = df.filter(col("email_status") == "Invalid")
print("valid_records:",valid_df.count())
print("rejected_records:",rejected_df.count())

valid_records: 22
rejected_records: 3


In [0]:
# Task 8: Save Silver and rejected data as Delta tables
valid_df.write.format("delta")\
    .mode("overwrite")\
    .saveAsTable("customer_pipeline.customer_data.silver_customers")

rejected_df.write.format("delta")\
    .mode("overwrite")\
    .saveAsTable("customer_pipeline.customer_data.rejected_customers")

In [0]:
%sql
select * 
from customer_pipeline.customer_data.silver_customers

customer_id,customer_name,city,email,age,salary,email_status
1003,Priya,Hyderabad,priya@gmail.com,24.0,46000.0,Valid
1015,Ajay,Bangalore,ajay@gmail.com,29.0,53000.0,Valid
1022,Swathi,Bangalore,swathi@gmail.com,24.0,0.0,Valid
1025,Asha,Chennai,asha@gmail.com,29.0,54000.0,Valid
1002,Arun,Chennai,arun@gmail.com,28.0,51000.0,Valid
1007,Vijay,Bangalore,vijay@gmail.com,32.0,58000.0,Valid
1008,Anita,Delhi,anita@gmail.com,26.0,45000.0,Valid
1011,Manoj,Bangalore,manoj@gmail.com,35.0,70000.0,Valid
1024,Varun,Delhi,varun@gmail.com,27.0,50000.0,Valid
1001,Rahul,Bangalore,rahul@gmail.com,25.0,42000.0,Valid


In [0]:
%sql
select * 
from customer_pipeline.customer_data.rejected_customers

customer_id,customer_name,city,email,age,salary,email_status
1021,Naveen,Pune,naveen@,30.0,56000.0,Invalid
1018,Geetha,Hyderabad,null,26.0,44000.0,Invalid
1020,Deepa,Mumbai,deepagmail.com,25.0,41000.0,Invalid
